#Paquetes necesarios

In [1]:
import cv2  
import math 

from ultralytics import YOLO

In [4]:
from collections import defaultdict
import numpy as np

# Carga del modelo, descarga en disco si no está presente en la carpeta
model = YOLO('yolo11n.pt') #Contenedores

# Etiqueta de las distintas clases
classNames = ["person", "bicycle", "car"]


# Captura desde la webcam
vid = cv2.VideoCapture("C0142.MP4")
track_history = defaultdict(lambda: [])

# Para guardar el vídeo procesado para ello necesitamos el códec, los fps y el tamaño del vídeo original
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = vid.get(cv2.CAP_PROP_FPS)
width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Guardar el vídeo procesado
out = cv2.VideoWriter("C0142_procesado.mp4", fourcc, fps, (width, height))

# Inicializamos los contadores y los sets para almacenar los IDs contados
contadores = {"person": 0, "bicycle": 0, "car": 0}
ids_contados = {"person": set(), "bicycle": set(), "car": set()}

while(True):      
    # fotograma a fotograma
    ret, img = vid.read()
  
    # si hay imagen válida
    if ret:  
        # Seguimiento, con persistencia entre fotogramas
        results = model.track(img, persist=True, classes = [0,1,2])
        
        # Para cada detección
        for r in results:
            boxes = r.boxes

            for box in boxes:
                # Contenedor
                x1, y1, x2, y2 = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Aplicar blur
                roi = img[y1:y2, x1:x2]
                roi_blur = cv2.GaussianBlur(roi, (51, 51), 0)
                img[y1:y2, x1:x2] = roi_blur

                # Etiqueta de seguimiento
                if box.id is not None:
                    track_id = str(int(box.id[0].tolist()))
                else:
                    track_id = ''
                
                # Confianza
                confidence = math.ceil((box.conf[0]*100))/100
                print("Confianza --->",confidence)

                # Clase
                cls = int(box.cls[0])
                print("Clase -->", classNames[cls])

       # Actualiza los contadores si el ID de seguimiento no ha sido contado previamente
                if box.id is not None:
                    nombre_clase = classNames[cls]
                    if track_id not in ids_contados[nombre_clase]:
                        ids_contados[nombre_clase].add(track_id)
                        contadores[nombre_clase] += 1

                # Convierte identificador numérico de clase a un color RGB
                escala = int((cls / len(classNames)) * 255 * 3)
                if escala >= 255*2:
                    R = 255
                    G = 255
                    B = escala - 255*2
                else:
                    if escala >= 255:
                        R = 255
                        G = escala - 255
                        B = 0
                    else:
                        R = escala
                        G = 0
                        B = 0

                # Dibuja el contenedor y clase
                cv2.rectangle(img, (x1, y1), (x2, y2), (R, G, B), 3)
                cv2.putText(img, track_id + ' ' + classNames[cls] , [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)

       # Dibuja los contadores en la esquina superior izquierda
        cv2.putText(img, f'Personas: {contadores["person"]}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(img, f'Bicicletas: {contadores["bicycle"]}', (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
        cv2.putText(img, f'Coches: {contadores["car"]}', (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

       # Guardar el fotograma procesado en el vídeo de salida
        out.write(img)

        # Muestra fotograma
        cv2.imshow('Vid', img)
    
    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break
  
# Libera el objeto de captura
vid.release()
# Libera el objeto de escritura de vídeo
out.release()
# Destruye ventanas
cv2.destroyAllWindows()



0: 384x640 1 car, 152.5ms
Speed: 8.1ms preprocess, 152.5ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.85
Clase --> car

0: 384x640 1 car, 196.9ms
Speed: 11.1ms preprocess, 196.9ms inference, 4.3ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.82
Clase --> car

0: 384x640 1 car, 160.1ms
Speed: 8.8ms preprocess, 160.1ms inference, 3.2ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.82
Clase --> car

0: 384x640 1 car, 160.0ms
Speed: 5.8ms preprocess, 160.0ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.84
Clase --> car

0: 384x640 1 car, 187.7ms
Speed: 8.9ms preprocess, 187.7ms inference, 5.2ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.82
Clase --> car

0: 384x640 1 car, 165.8ms
Speed: 10.3ms preprocess, 165.8ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.81
Clase --> car

0: 384x640 1 car, 196.3ms
Speed: 13.1ms pre

In [3]:
from ultralytics import YOLO
import cv2

model = YOLO("yolo11n.pt")

classNames = ["person", "bicycle", "car"]

cap = cv2.VideoCapture("C0142.MP4")

# obtener las caracteristicas del video
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# crear el video de salida
out = cv2.VideoWriter(
    "C0142_direccion.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

# linea roja para coches y bicicletas
linea_vehiculos_y = int(height * 430 / 928)
linea_vehiculos_x1 = int(width * 500 / 1650)
linea_vehiculos_x2 = width - 5

# linea azul para personas
# esta linea atraviesa la acera en lugar de seguir su direccion
# linea azul para personas
persona_x1 = int(width * 450 / 1650)
persona_y1 = int(height * 325 / 928)

persona_x2 = int(width * 500 / 1650)
persona_y2 = int(height * 390 / 928)

# margen de seguridad
margen = 12

# contadores de direccion
flujo = {
    "person": {"acercandose": 0, "alejandose": 0},
    "bicycle": {"acercandose": 0, "alejandose": 0},
    "car": {"acercandose": 0, "alejandose": 0}
}

# guardar el ultimo lado de cada objeto
posiciones = {}

# guardar la ultima posicion de las personas
posiciones_personas = {}

# evitar contar dos veces el mismo objeto
contados = set()

while cap.isOpened():
    ret, img = cap.read()

    if not ret:
        break

    # detectar y seguir los objetos
    results = model.track(
        img,
        persist=True,
        classes=[0, 1, 2],
        verbose=False
    )

    for box in results[0].boxes:

        # comprobar que el objeto tiene identificador
        if box.id is None:
            continue

        # obtener coordenadas, identificador y clase
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        track_id = int(box.id[0])
        cls = int(box.cls[0])
        nombre = classNames[cls]

        # utilizar el centro inferior de la caja
        cx = (x1 + x2) // 2
        cy = y2

        # analizar coches y bicicletas
        if nombre in ["car", "bicycle"]:

            # comprobar si esta por encima o debajo de la linea roja
            if linea_vehiculos_x1 <= cx <= linea_vehiculos_x2:

                if cy < linea_vehiculos_y - margen:
                    lado_actual = -1
                elif cy > linea_vehiculos_y + margen:
                    lado_actual = 1
                else:
                    lado_actual = 0

                # guardar la primera posicion conocida
                if track_id not in posiciones:
                    if lado_actual != 0:
                        posiciones[track_id] = lado_actual

                # comprobar si ha cruzado la linea
                elif track_id not in contados:

                    lado_anterior = posiciones[track_id]

                    if lado_actual != 0:

                        # de arriba hacia abajo: se acerca
                        if lado_anterior == -1 and lado_actual == 1:
                            flujo[nombre]["acercandose"] += 1
                            contados.add(track_id)

                        # de abajo hacia arriba: se aleja
                        elif lado_anterior == 1 and lado_actual == -1:
                            flujo[nombre]["alejandose"] += 1
                            contados.add(track_id)

                        # actualizar el ultimo lado conocido
                        posiciones[track_id] = lado_actual

        # analizar personas en la acera
        elif nombre == "person":

            # guardar la posicion inicial del peaton
            if track_id not in posiciones_personas:
                posiciones_personas[track_id] = (cx, cy)

            elif track_id not in contados:

                # recuperar la posicion anterior
                anterior_x, anterior_y = posiciones_personas[track_id]

                # calcular en que lado de la linea esta el peaton
                lado_anterior = (
                    (persona_x2 - persona_x1) * (anterior_y - persona_y1)
                    - (persona_y2 - persona_y1) * (anterior_x - persona_x1)
                )

                lado_actual = (
                    (persona_x2 - persona_x1) * (cy - persona_y1)
                    - (persona_y2 - persona_y1) * (cx - persona_x1)
                )

                # comprobar si ha cambiado de lado
                if lado_anterior * lado_actual < 0:

                    # comprobar que el cruce se produce cerca del segmento
                    medio_x = (anterior_x + cx) // 2
                    medio_y = (anterior_y + cy) // 2

                    cerca_linea = (
                        min(persona_x1, persona_x2) - 30 <= medio_x <= max(persona_x1, persona_x2) + 30
                        and min(persona_y1, persona_y2) - 30 <= medio_y <= max(persona_y1, persona_y2) + 30
                    )

                    if cerca_linea:

                        # de un lado hacia el otro: se acerca
                        if lado_anterior > 0 and lado_actual < 0:
                            flujo["person"]["alejandose"] += 1

                        else:
                            flujo["person"]["acercandose"] += 1

                        contados.add(track_id)

                # actualizar la posicion anterior
                posiciones_personas[track_id] = (cx, cy)

        # dibujar la deteccion y el identificador
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

        cv2.putText(
            img,
            f"{nombre} ID:{track_id}",
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            2
        )

    # dibujar la linea roja de vehiculos
    cv2.line(
        img,
        (linea_vehiculos_x1, linea_vehiculos_y),
        (linea_vehiculos_x2, linea_vehiculos_y),
        (0, 0, 255),
        3
    )

    # dibujar los limites del margen de seguridad
    cv2.line(
        img,
        (linea_vehiculos_x1, linea_vehiculos_y - margen),
        (linea_vehiculos_x2, linea_vehiculos_y - margen),
        (0, 255, 255),
        1
    )

    cv2.line(
        img,
        (linea_vehiculos_x1, linea_vehiculos_y + margen),
        (linea_vehiculos_x2, linea_vehiculos_y + margen),
        (0, 255, 255),
        1
    )

    # dibujar la linea azul de personas
    cv2.line(
        img,
        (persona_x1, persona_y1),
        (persona_x2, persona_y2),
        (255, 0, 0),
        3
    )

    # mostrar los contadores
    y = 30

    for nombre in classNames:
        texto = (
            f"{nombre}: "
            f"acercandose {flujo[nombre]['acercandose']} "
            f"alejandose {flujo[nombre]['alejandose']}"
        )

        cv2.putText(
            img,
            texto,
            (10, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 0, 255),
            2
        )

        y += 25

    # guardar y mostrar el fotograma
    out.write(img)
    cv2.imshow("Analisis de direccion", img)

    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break

# liberar recursos
cap.release()
out.release()
cv2.destroyAllWindows()

# mostrar los resultados finales
print("Resultados del flujo:")

for nombre in classNames:
    print(nombre, flujo[nombre])

Resultados del flujo:
person {'acercandose': 0, 'alejandose': 0}
bicycle {'acercandose': 0, 'alejandose': 0}
car {'acercandose': 0, 'alejandose': 0}


In [ ]:

import json
import random
import shutil
from pathlib import Path

# localizar el dataset original
posibles_rutas = [Path("../matriculas"), Path("matriculas")]
raiz_dataset = None

for ruta in posibles_rutas:
    if (ruta / "images").is_dir() and (ruta / "annotations").is_dir():
        raiz_dataset = ruta
        break

if raiz_dataset is None:
    raise FileNotFoundError("No se encuentra el dataset original")

imagenes_dir = raiz_dataset / "images"
anotaciones_dir = raiz_dataset / "annotations"
salida_dir = raiz_dataset / "dataset_yolo"

print("Dataset original:", raiz_dataset.resolve())

# buscar imagenes y json
extensiones = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

imagenes = {}
for ext in extensiones:
    for archivo in imagenes_dir.glob("*" + ext):
        imagenes[archivo.stem] = archivo

anotaciones = {
    archivo.stem: archivo
    for archivo in anotaciones_dir.glob("*.json")
}

# comprobar que cada imagen tenga su json
comunes = sorted(set(imagenes) & set(anotaciones))

print("\nImágenes encontradas:", len(imagenes))
print("JSON encontrados:", len(anotaciones))
print("Parejas encontradas:", len(comunes))
print("Imágenes sin JSON:", len(set(imagenes) - set(anotaciones)))
print("JSON sin imagen:", len(set(anotaciones) - set(imagenes)))

# preparar las anotaciones validas
datos = []
json_invalidos = 0
anotaciones_descartadas = 0

for nombre in comunes:
    try:
        with open(anotaciones[nombre], "r", encoding="utf-8") as f:
            contenido = json.load(f)

        if not isinstance(contenido, list):
            json_invalidos += 1
            continue

        etiquetas = []

        for objeto in contenido:
            if not isinstance(objeto, dict):
                anotaciones_descartadas += 1
                continue

            rect = objeto.get("rectangle")

            if not isinstance(rect, list) or len(rect) != 4:
                anotaciones_descartadas += 1
                continue

            try:
                x, y, w, h = map(float, rect)
            except (TypeError, ValueError):
                anotaciones_descartadas += 1
                continue

            if not all(0 <= v <= 1 for v in [x, y, w, h]) or w <= 0 or h <= 0:
                anotaciones_descartadas += 1
                continue

            etiquetas.append(f"0 {x:.10f} {y:.10f} {w:.10f} {h:.10f}")

        if etiquetas:
            datos.append((imagenes[nombre], etiquetas))

    except (json.JSONDecodeError, OSError, TypeError):
        json_invalidos += 1

# mezclar y dividir el dataset
random.seed(1)
random.shuffle(datos)

total = len(datos)
n_train = int(total * 0.70)
n_val = int(total * 0.20)

conjuntos = {
    "train": datos[:n_train],
    "val": datos[n_train:n_train + n_val],
    "test": datos[n_train + n_val:]
}

# crear las carpetas de salida
if salida_dir.exists():
    shutil.rmtree(salida_dir)

for conjunto, elementos in conjuntos.items():
    (salida_dir / "images" / conjunto).mkdir(parents=True, exist_ok=True)
    (salida_dir / "labels" / conjunto).mkdir(parents=True, exist_ok=True)

    for imagen, etiquetas in elementos:
        shutil.copy2(imagen, salida_dir / "images" / conjunto / imagen.name)

        etiqueta_file = salida_dir / "labels" / conjunto / (imagen.stem + ".txt")

        with open(etiqueta_file, "w", encoding="utf-8") as f:
            f.write("\n".join(etiquetas))

# crear configuracion de YOLO
yaml = f"""path: {salida_dir.resolve().as_posix()}
train: images/train
val: images/val
test: images/test

nc: 1
names: ['matricula']
"""

with open(salida_dir / "data.yaml", "w", encoding="utf-8") as f:
    f.write(yaml)

# mostrar resultados
print("\nDataset preparado correctamente")
print("Imágenes válidas:", total)
print("JSON inválidos:", json_invalidos)
print("Anotaciones descartadas:", anotaciones_descartadas)

for conjunto, elementos in conjuntos.items():
    print(f"{conjunto}: {len(elementos)} imágenes")

print("\nDataset guardado en:", salida_dir.resolve())


Dataset original: C:\Users\David\OneDrive\Documents\VC_P4\VC_P4\matriculas

Imágenes encontradas: 1456
JSON encontrados: 1455
Parejas encontradas: 1455
Imágenes sin JSON: 1
JSON sin imagen: 0


KeyboardInterrupt: 

In [22]:

import torch
from ultralytics import YOLO
from pathlib import Path

# rutas
dataset_dir = Path("../matriculas/dataset_yolo")
modelo_path = Path("yolo11n.pt")

# comprobar que existen los archivos
if not dataset_dir.exists():
    raise FileNotFoundError("No se encuentra el dataset")

if not modelo_path.exists():
    raise FileNotFoundError("No se encuentra yolo11n.pt")

# seleccionar dispositivo
device = 0 if torch.cuda.is_available() else "cpu"

print("Dispositivo:", torch.cuda.get_device_name(0) if device == 0 else "CPU")

# cargar modelo preentrenado
model = YOLO(str(modelo_path))

# entrenar detector de matriculas
resultados = model.train(
    data=str(dataset_dir / "data.yaml"),
    epochs=100,
    patience=20,
    imgsz=640,
    batch=8,
    device=device,
    workers=0,
    project="../matriculas/runs",
    name="train_matriculas",
    exist_ok=True,
    plots=True
)


Dispositivo: NVIDIA GeForce RTX 2060
New https://pypi.org/project/ultralytics/8.4.198 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.191  Python-3.9.21 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=..\matriculas\dataset_yolo\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=

In [23]:

from pathlib import Path
from ultralytics import YOLO

# localizar el mejor modelo entrenado
candidatos = list(Path("..").rglob("train_matriculas/weights/best.pt"))

if not candidatos:
    candidatos = list(Path(".").rglob("train_matriculas/weights/best.pt"))

if not candidatos:
    raise FileNotFoundError("No se encuentra best.pt")

modelo_path = candidatos[0]

# localizar el dataset
dataset_dir = Path("../matriculas/dataset_yolo")

if not dataset_dir.exists():
    dataset_dir = Path("matriculas/dataset_yolo")

print("Modelo:", modelo_path.resolve())
print("Dataset:", dataset_dir.resolve())

# cargar el mejor modelo
model = YOLO(str(modelo_path))

# evaluar con las imagenes de test
resultados_test = model.val(
    data=str(dataset_dir / "data.yaml"),
    split="test",
    imgsz=640,
    batch=8,
    device=0,
    plots=True,
    project=str(dataset_dir.parent / "runs"),
    name="test_matriculas"
)


Modelo: F:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\P4\runs\matriculas\runs\train_matriculas\weights\best.pt
Dataset: F:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\matriculas\dataset_yolo
Ultralytics 8.4.191  Python-3.9.21 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
YOLO11n summary (fused): 101 layers, 2,597,051 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.1±0.0 ms, read: 952.5±468.9 MB/s, size: 2168.7 KB)
val: Scanning F:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\matriculas\dataset_yolo\labels\test... 146 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 146/146 90.5it/s 1.6s0.4s
val: F:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\matriculas\dataset_yolo\images\test\20261031_191028.jpg: corrupt JPEG restored and saved
val: F:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\matriculas\dataset_yolo\images\test\PXL_20261101_1420107

In [ ]:

import cv2
from pathlib import Path
from ultralytics import YOLO

# video de entrada
video_path = Path("C0142.MP4")

# cargar modelo
model = YOLO(r"C:\Users\David\OneDrive\Documents\VC_P4\VC_P4\P4\runs\matriculas\runs\train_matriculas\weights\best.pt")

# abrir video
cap = cv2.VideoCapture(str(video_path))

if not cap.isOpened():
    raise RuntimeError("No se pudo abrir el video")

fps = cap.get(cv2.CAP_PROP_FPS)
ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# crear video de salida
salida_path = video_path.parent / (video_path.stem + "_matriculas.mp4")

writer = cv2.VideoWriter(
    str(salida_path),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (ancho, alto)
)

# procesar video
total_detecciones = 0
fotograma = 0

while True:
    ret, frame = cap.read()

    if not ret:
        break

    # detectar matriculas
    resultados = model.predict(
        frame,
        imgsz=640,
        conf=0.25,
        verbose=False
    )

    # dibujar detecciones
    for box in resultados[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        confianza = float(box.conf[0])

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        cv2.putText(
            frame,
            f"Matricula {confianza:.2f}",
            (x1, max(y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2
        )

        total_detecciones += 1

    writer.write(frame)
    fotograma += 1

    if fotograma % 100 == 0:
        print(f"Procesados: {fotograma}/{total_frames}")

# liberar recursos
cap.release()
writer.release()

print("\nProcesamiento terminado")
print("Fotogramas:", fotograma)
print("Detecciones totales:", total_detecciones)
print("Video generado:", salida_path.resolve())


Procesados: 100/2832
Procesados: 200/2832
Procesados: 300/2832
Procesados: 400/2832
Procesados: 500/2832
Procesados: 600/2832
Procesados: 700/2832
Procesados: 800/2832
Procesados: 900/2832
Procesados: 1000/2832
Procesados: 1100/2832
Procesados: 1200/2832
Procesados: 1300/2832
Procesados: 1400/2832
Procesados: 1500/2832
Procesados: 1600/2832
Procesados: 1700/2832
Procesados: 1800/2832
Procesados: 1900/2832
Procesados: 2000/2832
Procesados: 2100/2832
Procesados: 2200/2832
Procesados: 2300/2832
Procesados: 2400/2832
Procesados: 2500/2832
Procesados: 2600/2832
Procesados: 2700/2832
Procesados: 2800/2832

Procesamiento terminado
Fotogramas: 2832
Detecciones totales: 2234
Video generado: F:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\P4\C0142_matriculas.mp4


In [10]:
import cv2
import math
import csv
import numpy as np
import easyocr
from collections import defaultdict
from ultralytics import YOLO

# Carga de los modelos
model = YOLO('yolo11n.pt')  # Personas, bicicletas y coches
model_matriculas = YOLO(r"C:\Users\David\OneDrive\Documents\VC_P4\VC_P4\P4\runs\matriculas\runs\train_matriculas\weights\best.pt")

# Inicializamos EasyOCR para reconocer las matrículas
reader = easyocr.Reader(['es', 'en'], gpu=False)

# Etiqueta de las distintas clases
classNames = ["person", "bicycle", "car"]

# Captura del vídeo
vid = cv2.VideoCapture("C0142.MP4")

if not vid.isOpened():
    raise Exception("No se ha podido abrir el vídeo")

track_history = defaultdict(lambda: [])

# Información del vídeo original
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = vid.get(cv2.CAP_PROP_FPS)
width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))

if fps <= 0:
    fps = 30

# Guardar el vídeo procesado
out = cv2.VideoWriter("C0142_procesado_Ocr.mp4", fourcc, fps, (width, height))

# Archivo CSV para almacenar los resultados
archivo_csv = open("C0142_resultados_Ocr.csv", "w", newline="", encoding="utf-8")
campos = [
    "fotograma", "tipo_objeto", "confianza",
    "identificador_tracking", "x1", "y1", "x2", "y2",
    "matricula_en_su_caso", "confianza_matricula",
    "mx1", "my1", "mx2", "my2", "texto_matricula"
]

escritor = csv.DictWriter(archivo_csv, fieldnames=campos)
escritor.writeheader()

# Inicializamos los contadores y los sets para almacenar los IDs contados
contadores = {"person": 0, "bicycle": 0, "car": 0}
ids_contados = {"person": set(), "bicycle": set(), "car": set()}

# Contador de fotogramas
num_fotograma = 0

while(True):

    # Fotograma a fotograma
    ret, img = vid.read()

    # Si hay imagen válida
    if ret:

        num_fotograma += 1

        # Seguimiento de personas, bicicletas y coches
        results = model.track(img, persist=True, classes=[0, 1, 2], verbose=False)

        # Detección de matrículas
        results_matriculas = model_matriculas.predict(img, conf=0.25, verbose=False)

        # Almacenamos las matrículas detectadas
        matriculas = []

        for r_mat in results_matriculas:
            for box_mat in r_mat.boxes:

                # Coordenadas de la matrícula
                mx1, my1, mx2, my2 = box_mat.xyxy[0].cpu().numpy()
                mx1, my1, mx2, my2 = int(mx1), int(my1), int(mx2), int(my2)

                # Evitamos coordenadas fuera de la imagen
                mx1 = max(0, mx1)
                my1 = max(0, my1)
                mx2 = min(width, mx2)
                my2 = min(height, my2)

                # Confianza de la detección de matrícula
                confianza_mat = float(box_mat.conf[0])

                # Recortamos la matrícula de la imagen original
                roi_mat = img[my1:my2, mx1:mx2]

                # Reconocimiento del texto de la matrícula
                texto_mat = ""
                confianza_ocr = 0.0

                # Aumentamos la resolución al doble
                roi_mat = cv2.resize(roi_mat, None, fx=2, fy=2,
                interpolation=cv2.INTER_CUBIC)

                #Convertimos a escala de grises
                roi_mat = cv2.cvtColor(roi_mat, cv2.COLOR_BGR2GRAY)

                #Mejoramos el contraste
                clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
                roi_mat = clahe.apply(roi_mat)

                if roi_mat.size > 0:
                    resultado_ocr = reader.readtext(roi_mat, allowlist='0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ')

                    if len(resultado_ocr) > 0:
                        mejor_resultado = max(resultado_ocr, key=lambda x: x[2])
                        texto_mat = mejor_resultado[1].replace(" ", "").upper()
                        confianza_ocr = float(mejor_resultado[2])

                # Guardamos la información de la matrícula
                matriculas.append({
                    "x1": mx1,
                    "y1": my1,
                    "x2": mx2,
                    "y2": my2,
                    "confianza": confianza_mat,
                    "confianza_ocr": confianza_ocr,
                    "texto": texto_mat
                })

        # Para cada detección de personas, bicicletas y coches
        for r in results:

            boxes = r.boxes

            for box in boxes:

                # Contenedor
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Evitamos coordenadas fuera de la imagen
                x1 = max(0, x1)
                y1 = max(0, y1)
                x2 = min(width, x2)
                y2 = min(height, y2)

                # Confianza
                confidence = float(box.conf[0])
                print("Confianza --->", round(confidence, 2))

                # Clase
                cls = int(box.cls[0])
                nombre_clase = classNames[cls]
                print("Clase -->", nombre_clase)

                # Etiqueta de seguimiento
                if box.id is not None:
                    track_id = str(int(box.id[0]))
                else:
                    track_id = ""

                # Inicializamos la información de la matrícula
                matricula_asociada = None

                # Asociamos matrículas únicamente a los coches
                if cls == 2:

                    for mat in matriculas:

                        # Centro de la matrícula
                        centro_x = (mat["x1"] + mat["x2"]) // 2
                        centro_y = (mat["y1"] + mat["y2"]) // 2

                        # Comprobamos si está dentro del coche
                        if (x1 <= centro_x <= x2 and
                            y1 <= centro_y <= y2):

                            # Si hay varias, seleccionamos la de mayor
                            # confianza de detección
                            if (matricula_asociada is None or
                                mat["confianza"] > matricula_asociada["confianza"]):
                                matricula_asociada = mat

                # Actualiza los contadores si el ID no se ha contado previamente
                if box.id is not None:

                    if track_id not in ids_contados[nombre_clase]:
                        ids_contados[nombre_clase].add(track_id)
                        contadores[nombre_clase] += 1

                # Información de matrícula para el CSV
                if matricula_asociada is not None:

                    tiene_matricula = "Si"
                    confianza_matricula = matricula_asociada["confianza"]
                    mx1 = matricula_asociada["x1"]
                    my1 = matricula_asociada["y1"]
                    mx2 = matricula_asociada["x2"]
                    my2 = matricula_asociada["y2"]
                    texto_matricula = matricula_asociada["texto"]

                else:

                    tiene_matricula = "No"
                    confianza_matricula = ""
                    mx1 = ""
                    my1 = ""
                    mx2 = ""
                    my2 = ""
                    texto_matricula = ""

                # Escribimos la detección en el CSV
                escritor.writerow({
                    "fotograma": num_fotograma,
                    "tipo_objeto": nombre_clase,
                    "confianza": confidence,
                    "identificador_tracking": track_id,
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,
                    "matricula_en_su_caso": tiene_matricula,
                    "confianza_matricula": confianza_matricula,
                    "mx1": mx1,
                    "my1": my1,
                    "mx2": mx2,
                    "my2": my2,
                    "texto_matricula": texto_matricula
                })

                # Convierte identificador numérico de clase a un color RGB
                escala = int((cls / len(classNames)) * 255 * 3)

                if escala >= 255*2:
                    R = 255
                    G = 255
                    B = escala - 255*2

                else:
                    if escala >= 255:
                        R = 255
                        G = escala - 255
                        B = 0
                    else:
                        R = escala
                        G = 0
                        B = 0

                # Dibuja el contenedor y clase
                cv2.rectangle(img, (x1, y1), (x2, y2), (R, G, B), 3)

                cv2.putText(
                    img,
                    track_id + ' ' + nombre_clase,
                    [x1, y1],
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255, 0, B),
                    2
                )

                # Si el objeto tiene matrícula, mostramos su información
                if matricula_asociada is not None:

                    texto = texto_matricula

                    if texto == "":
                        texto = "Matricula no reconocida"

                    cv2.rectangle(
                        img,
                        (mx1, my1),
                        (mx2, my2),
                        (0, 255, 255),
                        2
                    )

                    cv2.putText(
                        img,
                        texto,
                        (mx1, max(20, my1 - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (0, 255, 255),
                        2
                    )

        # Dibuja los contadores en la esquina superior izquierda
        cv2.putText(img, f'Personas: {contadores["person"]}',
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        cv2.putText(img, f'Bicicletas: {contadores["bicycle"]}',
                    (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        cv2.putText(img, f'Coches: {contadores["car"]}',
                    (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        # Guardar el fotograma procesado en el vídeo de salida
        out.write(img)

        # Muestra el fotograma
        cv2.imshow('Vid', img)

    else:
        break

    # Detenemos al pulsar ESC
    if cv2.waitKey(20) == 27:
        break

# Libera el objeto de captura
vid.release()

# Libera el objeto de escritura de vídeo
out.release()

# Cerramos el CSV
archivo_csv.close()

# Destruye ventanas
cv2.destroyAllWindows()


Using CPU. Note: This module is much faster with a GPU.


Confianza ---> 0.84
Clase --> car


c:\Users\David\anaconda3\envs\VC_P4\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Confianza ---> 0.81
Clase --> car
Confianza ---> 0.82
Clase --> car
Confianza ---> 0.84
Clase --> car
Confianza ---> 0.82
Clase --> car
Confianza ---> 0.81
Clase --> car
Confianza ---> 0.8
Clase --> car
Confianza ---> 0.79
Clase --> car
Confianza ---> 0.78
Clase --> car
Confianza ---> 0.81
Clase --> car
Confianza ---> 0.8
Clase --> car
Confianza ---> 0.78
Clase --> car
Confianza ---> 0.79
Clase --> car
Confianza ---> 0.79
Clase --> car
Confianza ---> 0.78
Clase --> car
Confianza ---> 0.77
Clase --> car
Confianza ---> 0.78
Clase --> car
Confianza ---> 0.78
Clase --> car
Confianza ---> 0.77
Clase --> car
Confianza ---> 0.78
Clase --> car
Confianza ---> 0.75
Clase --> car
Confianza ---> 0.73
Clase --> car
Confianza ---> 0.74
Clase --> car
Confianza ---> 0.72
Clase --> car
Confianza ---> 0.7
Clase --> car
Confianza ---> 0.71
Clase --> car
Confianza ---> 0.69
Clase --> car
Confianza ---> 0.67
Clase --> car
Confianza ---> 0.65
Clase --> car
Confianza ---> 0.63
Clase --> car
Confianza ---> 0.